# Rung 06b — is epoch 1 the better checkpoint? (T7)

Pre-registered in `local/specs/vit-lora/spec.md` §"ADDENDUM T7" **before running**.

`checkpoint-selection-vs-number` measured that `number`'s margin over the trivial floor
is best at **epoch 1** in BOTH arms, while `acc_OOD` selection took **epoch 2**. This
evaluates epoch 1 of both arms on the full 6252, same protocol as `eval_best`.

**Symmetric on purpose:** the decay appears in both arms, so evaluating only rung 06
could not separate "epoch 1 is better" from "epoch 1 of the ViT arm is better".

🔴 This does **not** reopen the PARTIAL verdict (that A/B was epoch 2 vs epoch 2,
symmetric) and does **not** by itself change the shipped checkpoint — switching the
selection criterion is a second variable and would be its own comparison.


In [ ]:
# papermill parameters
SMOKE = False          # True -> 40 questions per arm, just to prove the wiring
RUN_TAG = "ep1_full"

In [ ]:
import json, logging, os, sys, time
from pathlib import Path
import pandas as pd

# `merge_checkpoint` shells out to the bare `swift` binary. A papermill kernel does NOT
# inherit the env's bin/ on PATH, so it must be put there explicitly -- and it must be
# THIS interpreter's bin, so the CLI and the kernel come from the same environment.
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")
print("swift on PATH:", (Path(_envbin) / "swift").exists(), "->", _envbin)

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
for p in (REPO / "src", EXP / "_models", REPO / "experiments/02-lora-sft/_models"):
    if p.is_dir():
        sys.path.insert(0, str(p))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

from frame.config import BaselineConfig
from frame.run import run_baseline
from frame import ledger, metrics
from lora_sft_train import LoRAConfig, merge_checkpoint

ARMS = {
    "b0_baseline": REPO / "experiments/02-lora-sft/runs/02_lora_sft_v1",
    "b1_vit_lora": REPO / "experiments/06-vit-lora/runs/06_vit_lora_v1",
}
CKPT = "checkpoint-860"     # epoch 1 in both arms (2580 steps / 3 epochs)

# The QA parquets live in different places on the pod and on a laptop. Resolve by
# LOOKING for them, and fail loudly if neither has them -- an eval on an empty data root
# is the classic silent zero.
DATA_ROOT = next(
    (d for d in (REPO / "external_data" / "orena-data", Path("/workspace/orena-data"))
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()),
    None,
)
assert DATA_ROOT is not None, "no frame/test.parquet under repo/external_data/orena-data nor /workspace/orena-data"
print("data_root:", DATA_ROOT)
print("repo:", REPO)
for a, d in ARMS.items():
    print(f"  {a}: {d.exists()}  {d}")

## Gate 0 — the merged weights must exist (or be merged with the arm's OWN exporter)

rung 06's `merged/checkpoint-860` already exists (17 GB). rung 02's does not — it is
merged here with **rung 02's own** `merge_checkpoint`, so the model leg of the
comparison cannot drift between arms.

In [ ]:
merged = {}
for arm, run_dir in ARMS.items():
    m = run_dir / "merged" / CKPT
    if m.is_dir() and any(m.iterdir()):
        print(f"OK    {arm}: merged already present -> {m}")
    else:
        ck = sorted((run_dir / "ckpt").glob(f"v0-*/{CKPT}"))
        assert ck, f"{arm}: no adapter at {run_dir}/ckpt/v0-*/{CKPT}"
        print(f"MERGE {arm}: {ck[0]}")
        cfg = LoRAConfig(exp_dir=run_dir.parents[1], run_name=run_dir.name)
        t0 = time.perf_counter()
        m = merge_checkpoint(cfg, ck[0])
        print(f"      merged in {time.perf_counter()-t0:.0f}s -> {m}")
    merged[arm] = m
merged

## Eval — full 6252, protocol IDENTICAL to `eval_best`

Same judge, same `max_pixels`, same seed, same `run_baseline`. Any deviation would
invalidate the comparison against the epoch-2 numbers already measured.

In [ ]:
reports = {}
for arm, model_path in merged.items():
    print(f"\n{'='*70}\n{arm}  <-  {model_path}\n{'='*70}")
    cfg = BaselineConfig(
        data_root=DATA_ROOT,
        model_path=model_path,
        out_dir=ARMS[arm],
        run_name=RUN_TAG,
        max_pixels=1280 * 720,
        seed=42,
        n_eval=40 if SMOKE else None,
    )
    t0 = time.perf_counter()
    reports[arm] = run_baseline(cfg)
    print(f"{arm}: done in {(time.perf_counter()-t0)/60:.1f} min")

## Score canonically + GATE 0 on gold coverage

The floor understates (and the margin therefore overstates) when gold is incomplete —
`template_floor` drops ungolded rows from the numerator but keeps them in the
denominator, warning only via `logger.warning`. Coverage is asserted, not trusted.

In [ ]:
gold = ledger.gold_from_frame_parquets(DATA_ROOT)
strats = {}
for arm, run_dir in ARMS.items():
    res = pd.read_csv(run_dir / RUN_TAG / "results.csv")
    missing = set(res["qID"]) - set(gold.dropna(subset=["answer"])["qID"])
    assert not missing, f"{arm}: GATE 0 -- {len(missing)} qIDs without gold; margins would be inflated"
    print(f"GATE 0 {arm}: gold {len(res)}/{len(res)} OK")
    metrics.assert_no_dup_qid(res); metrics.assert_ood_from_qid(res); metrics.assert_all_rows_grouped(res)
    s = metrics.stratified_report(res, gold=gold)
    metrics.assert_floors_vs_eval_set(s)
    strats[arm] = s
    if not SMOKE:
        ledger.register_run(run_dir / RUN_TAG, s,
                            experiment=run_dir.parents[1].name,
                            run=f"{run_dir.name}__{RUN_TAG}",
                            model=f"{arm} epoch 1 ({CKPT})", date="2026-07-18")
print("gates OK")

## The pre-registered read

Primary = `bucket_mean`. Secondary, always reported = `number` margin over the
template-aware floor, ID and OOD. **Never raw `acc_number`** (data card §3).

Epoch-2 references, already measured: rung 02 `bucket_mean` **0.5486**, rung 06
**0.5667**. The verdict table lives in the spec — a human reads it.

In [ ]:
EP2 = {"b0_baseline": 0.5486234489956542, "b1_vit_lora": 0.566707285167315}
rows = []
for arm, s in strats.items():
    bf = pd.DataFrame(s["by_format"]); num = bf[bf.answer_format == "number"].set_index("distribution")
    rows.append({
        "arm": arm,
        "bucket_mean_ep1": s["bucket_mean"],
        "bucket_mean_ep2": EP2[arm],
        "delta_ep1_minus_ep2": s["bucket_mean"] - EP2[arm],
        "margin_ID": s["margin_ID"], "margin_OOD": s["margin_OOD"],
        "number_margin_ID": num.loc["ID", "margin"] if "ID" in num.index else float("nan"),
        "number_margin_OOD": num.loc["OOD", "margin"] if "OOD" in num.index else float("nan"),
    })
out = pd.DataFrame(rows)
if not SMOKE:
    out.to_csv(EXP / "RESULTS_epoch1.csv", index=False)
print(out.to_string(index=False))